
**Nama:** Muhammad Rafly Romeo Nasution <br>
**Kelas:** 3KA25 <br>
**NPM:** 10123875

---



# MK Praktikum Unggulan Universitas Gunadarma
# Mata Kuliah: Praktikum Terapan Teori Graf (Tingkat 3)

---
# Pertemuan VII

In [6]:
import sys

In [7]:
class Graph:
    # Constructor
    def __init__(self, edges, n):

        # A list of lists to represent an adjacency list
        self.adjList = [[] for _ in range(n)]

        # Add edges to the directed graph
        for (source, dest, weight) in edges:
            self.adjList[source].append((dest, weight))

# Perform DFS on the graph and set the departure time of all vertices of the graph

In [8]:
def DFS(graph, v, discovered, departure, time):

    # Mark the current node as discovered
    discovered[v] = True

    # Set arrival time – not needed
    # time = time + 1

    # Do for every edge (v, u)
    for (u, w) in graph.adjList[v]:
        # if `u` is not yet discovered
        if not discovered[u]:
            time = DFS(graph, u, discovered, departure, time)

    # Ready to backtrack
    # Set departure time of vertex `v`
    departure[time] = v

    time = time + 1
    return time

In [9]:
# The function performs the topological sort on a given DAG and then finds
# the longest distance of all vertices from the given source by running one pass
# of the Bellman–Ford algorithm on edges of vertices in topological order
def findShortestDistance(graph, source, n):

    # `departure` stores the vertex number using departure time as an index
    departure = [-1] * n

    # To keep track of whether a vertex is discovered or not
    discovered = [False] * n
    time = 0

    # Perform DFS on all undiscovered vertices
    for i in range(n):
        if not discovered[i]:
            time = DFS(graph, i, discovered, departure, time)

    cost = [sys.maxsize] * n
    cost[source] = 0

    # Process the vertices in topological order, i.e., in order of their decreasing departure time in DFS
    for i in reversed(range(n)):

        # For each vertex in topological order, relax the cost of its adjacent vertices
        v = departure[i]

        # Edge from `v` to `u` having weight `w`
        for (u, w) in graph.adjList[v]:
            # if the distance to destination `u` can be shortened by
            # taking edge (v, u), update cost to the new lower value
            if cost[v] != sys.maxsize and cost[v] + w < cost[u]:
                cost[u] = cost[v] + w

    # Print shortest paths
    for i in range(n):
        print(f'dist({source}, {i}) = {cost[i]}')

In [10]:
if __name__ == '__main__':

    # List of graph edges as per the above diagram
    edges = [
        (0, 6, 2), (1, 2, -4), (1, 4, 1), (1, 6, 8), (3, 0, 3), (3, 4, 5),
        (5, 1, 2), (7, 0, 6), (7, 1, -1), (7, 3, 4), (7, 5, -4)
    ]

    # Total number of nodes in the graph (labelled from 0 to 7)
    n = 8

    # Build a graph from the given edges
    graph = Graph(edges, n)

    # Source vertex
    source = 7

    # Find the shortest distance of all vertices from the given source
    findShortestDistance(graph, source, n)

dist(7, 0) = 6
dist(7, 1) = -2
dist(7, 2) = -6
dist(7, 3) = 4
dist(7, 4) = -1
dist(7, 5) = -4
dist(7, 6) = 6
dist(7, 7) = 0



## Logika Program

Program ini mencari **jalur terpendek (shortest path) dari satu sumber ke semua node** pada graf berarah tanpa siklus (DAG — Directed Acyclic Graph). Algoritma yang dipakai adalah kombinasi **Topological Sort via DFS** lalu **satu kali relaksasi Bellman-Ford**.

---

### Langkah 1 — Bangun Graf

Kelas `Graph` menerima daftar edge berformat `(sumber, tujuan, bobot)` dan menyimpannya sebagai adjacency list. Hasilnya seperti ini untuk beberapa node:

```
node 7 -> [(0,6), (1,-1), (3,4), (5,-4)]
node 5 -> [(1,2)]
node 1 -> [(2,-4), (4,1), (6,8)]
...
```

---

### Langkah 2 — DFS untuk Topological Sort

Fungsi `DFS` menelusuri graf secara rekursif. Setiap node dicatat **waktu keluarnya (departure time)** — yaitu saat semua tetangganya sudah selesai diproses. Node yang selesai paling akhir berarti tidak bergantung pada siapapun, sehingga dia harus diproses pertama dalam topological order.

Array `departure` menyimpan `departure[time] = node`, sehingga kalau dibaca dari belakang (`reversed`) itulah urutan topologisnya.

---

### Langkah 3 — Relaksasi Jarak

Setelah urutan topologis didapat, program menelusuri setiap node sesuai urutan tersebut dan melakukan **relaksasi**:

```
jika cost[v] + bobot_edge < cost[u]:
    perbarui cost[u]
```

Karena ini DAG dan urutannya topologis, setiap node pasti sudah diproses sebelum tetangganya — sehingga cukup satu kali lintasan saja, berbeda dengan Dijkstra atau Bellman-Ford penuh yang butuh iterasi berulang.

Nilai awal semua `cost` adalah tak hingga (`sys.maxsize`), kecuali node sumber yang bernilai `0`.

---

### Struktur Graf

```
7 --(-1)--> 1 --(-4)--> 2
7 --(-4)--> 5 --( 2)--> 1
7 --( 4)--> 3 --( 3)--> 0 --( 2)--> 6
7 --( 6)--> 0
            1 --( 1)--> 4
            1 --( 8)--> 6
            3 --( 5)--> 4
```

---

### Output

```
dist(7, 0) = 6
dist(7, 1) = -2
dist(7, 2) = -6
dist(7, 3) = 4
dist(7, 4) = -1
dist(7, 5) = -4
dist(7, 6) = 6
dist(7, 7) = 0
```

Penjelasan beberapa jalur terpendeknya:

| Tujuan | Jalur | Total |
|--------|-------|-------|
| 0 | 7 → 0 (bobot 6) | 6 |
| 1 | 7 → 5 → 1 (-4 + 2) | -2 |
| 2 | 7 → 5 → 1 → 2 (-4 + 2 + -4) | -6 |
| 5 | 7 → 5 (-4) | -4 |
| 7 | sumber itu sendiri | 0 |

Node `2` punya jarak paling negatif karena jalur lewat node `5` dan `1` mengakumulasi bobot negatif, yang memang diperbolehkan di DAG (berbeda dengan Dijkstra yang tidak bisa menangani bobot negatif).



---


Copyright © Pengelola MK Praktikum Unggulan (Praktikum DGX), Universitas Gunadarma


https://www.praktikum-hpc.gunadarma.ac.id/